In [2]:
%pip install lightgbm
%pip install pandas
%pip install scikit-learn
%pip install tqdm
%pip install requests
%pip install nltk

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\erika\.pyenv\pyenv-win\versions\3.9.0\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\erika\.pyenv\pyenv-win\versions\3.9.0\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\erika\.pyenv\pyenv-win\versions\3.9.0\python.exe -m pip install --upgrade pip' command.


... (2 outputs left out)

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\erika\.pyenv\pyenv-win\versions\3.9.0\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
You should consider upgrading via the 'c:\Users\erika\.pyenv\pyenv-win\versions\3.9.0\python.exe -m pip install --upgrade pip' command.


In [ ]:
%pip install --upgrade ir_datasets

In [4]:
import pandas as pd

df_rows_for_msmarcodocs = pd.read_csv('training_data/msmarco-docs-lookup.tsv', sep='\t', names=['offset_trec', 'offset_tsv'])

In [5]:
result = df_rows_for_msmarcodocs.loc["D1088"]
print(result["offset_tsv"])

966093490


In [6]:
# Function to get the document using the byte offset
def get_document_by_offset(offset_tsv):
    # Step 2: Use the offset to read the line from the large TSV file
    file_path = 'training_data/msmarco/msmarco-docs.tsv'
    with open(file_path, 'r', encoding='utf-8') as file:
        file.seek(offset_tsv)
        line = file.readline()
        docid, url, title, body = line.strip().split('\t')
        document = {
            'docid': docid,
            'url': url,
            'title': title,
            'body': body
        }
        return document

In [7]:
x = get_document_by_offset(966093490)
print(x)

{'docid': 'D1088', 'url': 'http://www.americananthro.org/AdvanceYourCareer/Content.aspx?ItemNumber=1783', 'title': 'Careers in Anthropology', 'body': '"Home | Advance Your Career What are the job prospects for anthropologists? The anthropology job market is competitive. According to the United States Bureau of Labor Statistics, ""Employment of anthropologists and archaeologists is expected to grow four percent from 2014 to 2024. "" ( US Department of Labor ). The median pay for anthropologists and archaeologists in the United States was $61,220 in 2015. Citation: Bureau of Labor Statistics, U. S. Department of Labor, Occupational Outlook Handbook, 2016-17 Edition, Anthropologists and Archeologists,on the Internet at http://www.bls.gov/ooh/life-physical-and-social-science/anthropologists-and-archeologists.htm (visited October 25, 2016 ). Where are anthropologists working? Today\'s anthropologists do not just work in exotic locations. Anthropologists can be found in a surprising array of

In [28]:
import ir_datasets
from itertools import islice
from compute_features import PrecomputedDocumentFeatures
import nltk

nltk.download('wordnet')

def extract_features(group_docs, query):
    pdf = PrecomputedDocumentFeatures(group_docs)
    group_features = pdf.extract_query_features(query)
    normalized_features = pdf.normalize_features(group_features)
    return normalized_features


def prepare_msmarco_dataset(dataset, top_k):
    g = dataset.scoreddocs_iter()
    documents = list(islice(g, top_k))
    
    query_cache = {}
    for query in dataset.queries_iter():
        if query.query_id in query_cache:
            continue  # Skip if already cached
        for doc in documents:
            if query.query_id == doc.query_id:
                query_cache[doc.query_id] = query.text
                break  

    # Filter documents that have actual qrels
    qrel_dict = {(qrel.query_id, qrel.doc_id): qrel.relevance for qrel in dataset.qrels_iter()}
    filtered_documents = [doc for doc in documents if (doc.query_id, doc.doc_id) in qrel_dict]

    # Prepare DataFrames for features (X) and targets (y1, y2)
    # Prepare lists for features and targets
    features = []
    targets1 = []
    targets2 = []
    groups = []

    current_query_id = None
    current_group_size = 0
    group_docs = []

    for scoreddoc in filtered_documents:
        if current_query_id is None or scoreddoc.query_id != current_query_id: # if a new query_id appears, then the group is closed
            if current_group_size > 0: # filter 0 out for appending
                groups.append(current_group_size)
                query = query_cache[current_query_id]
                group_features = extract_features(group_docs, query)
                features.extend(group_features)

            # reset for a new query
            current_query_id = scoreddoc.query_id
            current_group_size = 0
            group_docs = []

        current_group_size += 1

        result = df_rows_for_msmarcodocs.loc[scoreddoc.doc_id]

        # Retrieve the document text using the offset
        document = get_document_by_offset(result["offset_tsv"])
        group_docs.append(document)

        # Add the target scores
        targets1.append(qrel_dict[(scoreddoc.query_id, scoreddoc.doc_id)])  # target1 from qrels
        targets2.append(scoreddoc.score)  # target2 from scoreddoc

    # Append the last group size
    if current_group_size > 0:
        groups.append(current_group_size)
        query = query_cache[current_query_id]
        group_features = extract_features(group_docs, query)
        features.extend(group_features)

    # Create DataFrames for features and targets
    X = pd.DataFrame(features, columns=[f"feature_{i}" for i in range(len(features[0]))])
    y = pd.DataFrame({'target1': targets1, 'target2': targets2})

    # Saving the information in three DataFrames: X, y, and groups
    X.to_csv("X.csv", index=False)
    y.to_csv("y.csv", index=False)
    pd.Series(groups, name="group").to_csv("groups.csv", index=False)

    return X, y, groups

dataset = ir_datasets.load("msmarco-document/trec-dl-2019/judged")
X, y, groups = prepare_msmarco_dataset(dataset, 10000) # qrels are sparsely labeled, not in the beginning; 4.3k scored docs

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\erika\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
c:\Users\erika\dev\MSE\ModernSearchEngines\retriever\compute_features.py:213: RuntimeWarning: invalid value encountered in divide
  return ((features - min_vals) / (max_vals - min_vals)).tolist()


Combine human annotated relevance score and automatic scores.

In [49]:

# Assign weights inversely proportional to their occurrence. 
# This means that less frequent (more relevant) categories get higher weights, while more frequent (less relevant) categories get lower weights.
weights = {
    0: 1,
    1: 59.4 / 28.3,
    2: 59.4 / 7.1,
    3: 59.4 / 5.2
}

# Apply weights to the relevance
y['weighted_relevance'] = y['target1'].map(weights)
y['combined_score'] = y['weighted_relevance'] + y['target2']
# Normalized
y['combined_score'] = (y['combined_score'] - y['combined_score'].min()) / (y['combined_score'].max() - y['combined_score'].min())



In [50]:
y

,target1,target2,weighted_relevance,combined_score
0,2,-4.69200,8.366197,0.762588
1,3,-4.81849,11.423077,0.958804
2,2,-4.82038,8.366197,0.753992
3,1,-5.06599,2.098940,0.317896
4,2,-5.09501,8.366197,0.735603
...,...,...,...,...
2556,0,-7.37940,1.000000,0.089408
2557,0,-7.38036,1.000000,0.089344
2558,0,-7.38048,1.000000,0.089336
2559,0,-7.40885,1.000000,0.087436


Do a train-test split.

In [52]:
# Calculate the indices for the train-test split based on the groups
train_indices = sum(groups[:38])
test_indices = sum(groups[38:])

# Split the data based on the calculated indices
X_train = X[:train_indices]
X_test = X[train_indices:]
y_train = y[:train_indices]
y_test = y[train_indices:]
groups_train = groups[:38]
groups_test = groups[38:]

GBM training

In [47]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\erika\.pyenv\pyenv-win\versions\3.9.0\python.exe -m pip install --upgrade pip' command.


In [57]:
import xgboost as xgb

# Define the data in DMatrix format
dtrain = xgb.DMatrix(X_train, label=y_train['combined_score'])
dtest = xgb.DMatrix(X_test, label=y_test['combined_score'])

# Set up the parameters for LambdaMART
params = {
    'objective': 'rank:ndcg',  
    'learning_rate': 0.1,
    'gamma': 1.0,
    'min_child_weight': 0.1,
    'max_depth': 6,
    'n_estimators': 1000,
    'eval_metric': 'ndcg'
}

# Initialize the XGBRanker
ranker = xgb.XGBRanker(**params)

# Fit the model
ranker.fit(X_train, y_train['target1'], group=groups_train)

# Predict on the test set
y_pred = ranker.predict(X_test)

# Print the evaluation results
print("Predictions on the test set:")
print(y_pred)


Predictions on the test set:
[ 0.05897565 -0.65676564 -0.7035871  -0.4656842  -0.84553    -0.83652264
 -0.84553    -0.27739894 -0.84553    -0.84553    -0.5586795  -0.41378286
 -0.34457532 -0.56742454 -0.84553    -0.18300812  0.36127692 -0.7035871
 -0.27471963  0.5478063  -0.7311587  -0.2446547  -0.84553    -0.84553
 -0.42822504 -0.84553    -0.84553    -0.84553    -0.6054914  -0.6054914
 -0.84553    -0.4192177   0.24131714 -0.6054914  -0.80422896 -0.6054914
 -0.7783537  -0.5215275  -0.78717124 -0.6361544  -0.84553    -0.84553
 -0.34457532 -0.31811935 -0.84553    -0.54057145  0.01977122 -0.84553
 -0.84553    -0.84553    -0.83998126 -0.634357   -0.84553    -0.7886011
 -0.84553    -0.47731823  0.08849619 -0.04412075 -0.4721761  -0.7783537
 -0.34457532 -0.4721761  -0.62880826 -0.84553    -0.83998126 -0.5215275
 -0.84553    -0.4721761  -0.34457532 -0.83652264  0.32268465 -0.34457532
 -0.7035871  -0.634357   -0.5285038   0.48605582  0.9697254  -0.20491166
  0.7644939  -0.33023313 -0.72044176 

In [ ]:
import requests

# Here instead of hostname you can also use the ip adresse Displayed in NordVPN
url = 'http://your-db-host:5000/query'

N = 1000  # Change this to the number of documents you need

# Update the SQL query to fetch the first N documents
query = f'SELECT * FROM documents LIMIT {N}'

# Basic authentication details
auth = ('mseproject', '***REMOVED***')

# Make the POST request to the Flask API with your query
response = requests.post(url, json={'query': query}, auth=auth)

# Check if the request was successful
if response.status_code == 200:
    # Parse the JSON response
    data = response.json()
    if data:
        #here i can access the result via the index "count" because i renamed it ^
        count = data
        print(f"Total documents count: {count}")
    else:
        print("No data returned.")
else:
    print(f"Error executing query: {response.status_code} - {response.text}")

****From the past: Test on Microsoft Dataset****

In [ ]:
import numpy as np
from collections import Counter
from typing import List, Dict
import pandas as pd
from bm25 import BM25S

class PrecomputedDocumentFeatures:
    
    def __init__(self, documents: List[dict]):
        self.documents = documents
        self.doc_count = len(documents)
        self.idf_cache = {}
        self.precomputed_features = []
        self.titles = []
        self.bodies = []
        self.urls = []
        self.whole_documents = []
        self._build_idf_cache()
        self._precompute_features()
        
    
    def _build_idf_cache(self): 
        """ 
        Pre-compute IDF for all terms in the corpus. 
        WARNING: IDF defined as: 1 divided by the number of documents containing the query terms.
        """
        term_doc_count = Counter()
        for doc in self.documents:
            content = doc['content']
            url = doc['url']
            outgoing_links = doc['outgoing_links']
            whole_document = f"{content} {url} {outgoing_links}".lower() # lower!
            unique_terms = set(whole_document.split())
            for term in unique_terms:
                term_doc_count[term] += 1
        
        for term, count in term_doc_count.items():
            self.idf_cache[term] = 1 / (count + 1)
    
    def compute_idf(self, term: str) -> float:
        """ Compute IDF for a given term using the pre-computed IDF cache. """
        return self.idf_cache.get(term, 1 / (1 + 1))  # Default IDF for unseen terms
    
    def stream_length(self, text: str) -> int:
        """ Determine the length of the text in terms of word count. """
        return len(text.split())
    
    def term_frequencies(self, text: str) -> Dict[str, int]:
        """ Compute term frequencies for a given text. """
        return Counter(text.split())
    
    def _trim_content(self, content: str) -> str:
        """ Trim 20% off the beginning and end of the content as body. """
        words = content.split()
        total_length = len(words)
        cut_off = int(total_length * 0.2)
        trimmed_words = words[cut_off: total_length - cut_off]
        return ' '.join(trimmed_words)
    
    def _precompute_features(self):
        """ Pre-compute features for all documents. """
        for doc in self.documents:
            content = doc['content'].lower()
            body = self._trim_content(content)
            title = doc['title'].lower()
            url = doc['url'].lower()
            outgoing_links = doc['outgoing_links'].lower()
            url_splitted = url.replace('/', ' ').replace('.', ' ').split()
            url_terms = ' '.join(url_splitted)
            outgoing_links_terms = ' '.join(outgoing_links.replace('/', ' ').replace('.', ' ').split())
            whole_document = f"{content} {url_terms} {outgoing_links_terms}"
            
            body_term_freq = self.term_frequencies(body)
            title_term_freq = self.term_frequencies(title)
            url_term_freq = self.term_frequencies(url_terms)
            whole_doc_term_freq = self.term_frequencies(whole_document)
            
            body_features = {
                'stream_length': self.stream_length(body),
                'term_freq': body_term_freq
            }
            
            title_features = {
                'stream_length': self.stream_length(title),
                'term_freq': title_term_freq
            }
            
            url_features = {
                'stream_length': len(url_splitted[2]), # only the main part of the url e.g. tuebingen from https://www.tuebingen.de
                'term_freq': url_term_freq
            }
            
            whole_doc_features = {
                'stream_length': self.stream_length(whole_document),
                'term_freq': whole_doc_term_freq
            }
            
            # For each document: precomputed features
            self.precomputed_features.append({
                'content': body_features,
                'title': title_features,
                'url': url_features,
                'whole_document': whole_doc_features,
                'num_slashes_url': url.count('/'),
                'num_char_url': len(url),
                'outlink_number': len(doc['outgoing_links'].split(",")),
                'pagerank': doc['pagerank'] * 100000 # too low to be in the MS dataset
            })

            self.titles.append(title.split())
            self.urls.append(url_splitted)
            self.whole_documents.append(whole_document.split())
            self.bodies.append(body.strip())
    
    def extract_query_features(self, query: str, order=False, bm25=False) -> pd.DataFrame:
        """ Extract query-specific features using pre-computed document features. """
        query_terms = query.lower().split()
        all_features = []
        
        part_order = {'content': 1, 'title': 3, 'url': 4, 'whole_document': 5}

        for doc_features in self.precomputed_features: # for each document
            features = {}
            for part in ['content', 'title', 'url', 'whole_document']:
                term_freq = doc_features[part]['term_freq']
                # Use the precomputed IDF values for the query terms
                idf_values = [self.compute_idf(term) for term in query_terms]
                
                query_term_freqs = [term_freq[term] for term in query_terms if term in term_freq]
                tf_idf_values = [term_freq[term] * idf for term, idf in zip(query_terms, idf_values) if term in term_freq]
                
                covered_query_terms = len(query_term_freqs)
                sum_term_freq = sum(query_term_freqs)
                min_term_freq = min(query_term_freqs, default=0)
                max_term_freq = max(query_term_freqs, default=0)
                mean_term_freq = round(sum(query_term_freqs) / len(query_term_freqs), 6) if query_term_freqs else 0.000000
                var_term_freq = round(sum((x - mean_term_freq) ** 2 for x in query_term_freqs) / len(query_term_freqs), 6) if query_term_freqs else 0.000000
                
                sum_tf_idf = round(sum(tf_idf_values), 6) if tf_idf_values else 0.000000
                min_tf_idf = round(min(tf_idf_values), 6) if tf_idf_values else 0.000000
                max_tf_idf = round(max(tf_idf_values), 6) if tf_idf_values else 0.000000
                mean_tf_idf = round(sum(tf_idf_values) / len(tf_idf_values), 6) if tf_idf_values else 0.000000
                var_tf_idf = round(sum((x - mean_tf_idf) ** 2 for x in tf_idf_values) / len(tf_idf_values), 6) if tf_idf_values else 0.000000
                
                # IDF feature for the query terms
                idf = round(sum(idf_values) / len(idf_values), 6) if idf_values else 0.000000
                
                feature_values = [
                    covered_query_terms,
                    covered_query_terms / len(query_terms),
                    doc_features[part]['stream_length'],
                    idf,
                    sum_term_freq,
                    min_term_freq,
                    max_term_freq,
                    mean_term_freq,
                    var_term_freq,
                    sum_tf_idf,
                    min_tf_idf,
                    max_tf_idf,
                    mean_tf_idf,
                    var_tf_idf,
                    doc_features['num_slashes_url'],
                    doc_features['num_char_url'],
                    doc_features['outlink_number'],
                    doc_features['pagerank']
                ]
                
                # Microsoft Dataset feature names
                feature_names = [
                    part_order[part],
                    part_order[part] + 5,
                    part_order[part] + 10,
                    part_order[part] + 15,
                    part_order[part] + 20,
                    part_order[part] + 25,
                    part_order[part] + 30,
                    part_order[part] + 35,
                    part_order[part] + 40,
                    part_order[part] + 70,
                    part_order[part] + 75,
                    part_order[part] + 80,
                    part_order[part] + 85,
                    part_order[part] + 90,
                    126,
                    127,
                    129,
                    130
                ]
                for name, value in zip(feature_names, feature_values):
                    features[name] = value
            all_features.append(features)
        
        if bm25:
            from multiprocessing import cpu_count
            cpu_count = cpu_count()
            query_split = query.split()
            bm25 = BM25S(self.titles, num_threads=cpu_count, method="bm25+")
            scores = bm25.get_scores(query_split)
            for i, features in enumerate(all_features):
                features[108] = scores[i]

            bm25 = BM25S(self.bodies, num_threads=cpu_count, method="bm25+")
            scores = bm25.get_scores(query_split)
            for i, features in enumerate(all_features):
                features[106] = scores[i]

            bm25 = BM25S(self.urls, num_threads=cpu_count, method="bm25+")
            scores = bm25.get_scores(query_split)
            for i, features in enumerate(all_features):
                features[109] = scores[i]

                
            bm25 = BM25S(self.whole_documents, num_threads=cpu_count, method="bm25+")
            scores = bm25.get_scores(query_split)
            for i, features in enumerate(all_features):
                features[110] = scores[i]

        df_results = pd.DataFrame(all_features)

        if order:
            df_results = df_results.reindex(sorted(df_results.columns, key=int), axis=1)
        
        return df_results
# Example usage
documents = count

query = "Hölderlin Tower on the Neckar"
import time 

start = time.time()
feature_extractor = PrecomputedDocumentFeatures(documents) # assume the documents are not lowercased, not NLP perf for now
query_features = feature_extractor.extract_query_features(query, order=True, bm25=True)
stop = time.time()
times = stop - start
query_features


In [ ]:
query_features[66]

In [ ]:

import pandas as pd
import numpy as np

def get_data(data_path, top_k = None):
    if top_k:
        dfs = {
            "train": pd.read_csv(f"{data_path}/train.txt", delimiter=" ", header=None).head(top_k),
            "vali": pd.read_csv(f"{data_path}/vali.txt", delimiter=" ", header=None).head(top_k),
            "test": pd.read_csv(f"{data_path}/test.txt", delimiter=" ", header=None).head(top_k),
        }
    else:
        dfs = {
            "train": pd.read_csv(f"{data_path}/train.txt", delimiter=" ", header=None),
            "vali": pd.read_csv(f"{data_path}/vali.txt", delimiter=" ", header=None),
            "test": pd.read_csv(f"{data_path}/test.txt", delimiter=" ", header=None),
        }


    for df in dfs.values():
        df.columns = np.arange(len(df.columns))
        df.drop(columns=df.columns[df.isna().all()].tolist(), inplace=True)  # delete columns where all data is missing

    split = {}
    split["X_train"] = dfs["train"].iloc[:, 1:]
    split["X_val"] = dfs["vali"].iloc[:, 1:]
    split["X_test"] = dfs["test"].iloc[:, 1:]

    y_train = dfs["train"].iloc[:, 0]
    y_val = dfs["vali"].iloc[:, 0]
    y_test = dfs["test"].iloc[:, 0]

    # Create group parameters for LightGBM
    group_train = dfs["train"].groupby(1).size().to_list()
    group_vali = dfs["vali"].groupby(1).size().to_list()
    group_test = dfs["test"].groupby(1).size().to_list()

    # Specify the columns to keep
    columns_to_keep = [1, 3, 4, 5, 6, 8, 9, 10, 11, 13, 14, 15, 16, 18, 19, 20, 21, 23, 24, 25, 26, 28, 29, 30, 31, 33, 34, 35, 36, 38, 39, 40, 41, 43, 44, 45, 71, 73, 74, 75, 76, 78, 79, 80, 81, 83, 84, 85, 86, 88, 89, 90, 91, 93, 94, 95, 106, 108, 109, 110, 126, 127, 129, 130] 
    # add ti

    for name, df in split.items():
        df = df.applymap(lambda x: x.split(":", 1)[-1])  # Get rid of irrelevant information
        df = df.astype(float)  # Convert data into float format
        df = df.drop(columns=1)  # Drop the query ID column
        df.columns = [i for i in range(1, 137)]  # Rename column indices
        df = df[columns_to_keep]  # Keep only specified columns
        split[name] = df

    return (
        split["X_train"],
        split["X_test"],
        split["X_val"],
        y_train,
        y_test,
        y_val,
        group_train,
        group_vali,
        group_test
    )

In [ ]:
# Load data
data_path = "training_data/Fold1"
X_train, X_test, X_val, y_train, y_test, y_val, group_train, group_vali, group_test = get_data(data_path)

In [ ]:
X_val[130]

In [ ]:
from multiprocessing import cpu_count
import lightgbm as lgb
import tqdm
cpu_count = cpu_count()
print(cpu_count)
total_boost_round = 1000 # over 1000 its just repeating

# Train the LightGBM model
lgb_ranker = lgb.LGBMRanker(
    objective='lambdarank',
    boosting_type='dart', # enhancement of gradient boosting decision tree
    num_leaves=31,
    learning_rate=0.05,
    n_estimators=total_boost_round,  # Maximum number of boosting rounds; epochs
    n_jobs=cpu_count
    
)

lgb_ranker.fit(
    X_train, y_train,
    group=group_train,
    eval_set=[(X_test, y_test)],
    eval_group=[group_test],
    eval_at=[10],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50), # early stopping not available with dart,
        lgb.log_evaluation(period=1, show_stdv=True),  # Log evaluation results
    ]
)

In [ ]:
# Predict relevance scores for the test set
y_pred = lgb_ranker.predict(query_features, pred_contrib=False)

In [ ]:
import pandas as pd
# Convert SHAP values to a DataFrame for easier interpretation
shap_df = pd.read_csv("shap.csv")

# Print the first few rows of the SHAP values DataFrame
print(shap_df.head())

In [ ]:
import shap
shap_values_no_bias = shap_df.drop(columns=["bias", "Unnamed: 0"])
shap_values = shap_values_no_bias.values

# Summary plot
shap.summary_plot(shap_values, feature_names=shap_values_no_bias.columns)

In [ ]:
%pip install shap
%pip install matplotlib

In [ ]:
# Save the model to a file
model_filename = 'lgb_ranker_model1.txt'
lgb_ranker.booster_.save_model(model_filename)

**Test on our data**

In [ ]:
import lightgbm as lgb
from lightgbm import LGBMRanker
# Load the model from the file
ranker = lgb.Booster(model_file='lgb_ranker_model1.txt')


In [ ]:
predictions = ranker.predict(query_features)

In [ ]:
predictions.shape

In [ ]:
# Get the indices of the top 10 predictions
top_10_indices = np.argsort(predictions)[-10:][::-1]
print("Indices of the top 10 predictions:", top_10_indices)

for i in top_10_indices:
    print(count[i])

In [ ]:
count[45]